In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [10]:
import numpy as np
from embedder import Embedder

# 1. Instanciar el embebedor local con el modelo descargado
embedder = Embedder("../models/Xenova/all-MiniLM-L6-v2")

# 2. Tu consulta de la Pregunta 1
query = "¿Cómo funciona la búsqueda aproximada de vecinos más cercanos?"
v_query = embedder.encode(query)

# 3. Buscar el documento de la lección 07 dentro de la lista 'documents'
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = next((doc for doc in documents if doc["filename"] == target_filename), None)

if target_doc:
    # 4. Generar el embedding del contenido de esa página
    v_doc = embedder.encode(target_doc["content"])
    
    # 5. Calcular la similitud (el modelo ya devuelve vectores normalizados, 
    # por lo que el producto punto es equivalente a la similitud de coseno)
    similarity = np.dot(v_query, v_doc)
    
    print(f"Similitud de coseno encontrada: {similarity:.2f}")
else:
    print(f"No se encontró el archivo {target_filename} en la lista de documentos.")


Similitud de coseno encontrada: 0.01


In [ ]:
# ==============================================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ==============================================================================

# 'numpy' es la librería estándar para cálculo numérico y manejo de vectores o matrices a alta velocidad.
# 'np' es el alias convencional que usamos para no tener que escribir 'numpy' a cada rato.
import numpy as np

# 'Embedder' es una clase personalizada dentro de tu archivo local 'embedder.py'.
# Se encarga de cargar el modelo ONNX y convertir texto legible en vectores matemáticos.
from embedder import Embedder

# 'GithubRepositoryDataReader' sirve para descargar y estructurar archivos de un repositorio de GitHub.
# 'chunk_documents' es una función utilitaria para cortar textos muy largos en bloques más chicos.
from gitsource import GithubRepositoryDataReader, chunk_documents

# 'Index' se utiliza para búsquedas clásicas por texto plano (coincidencia de palabras exactas).
# 'VectorSearch' se utiliza para búsquedas semánticas (coincidencia por significado matemático).
from minsearch import Index, VectorSearch


# ==============================================================================
# 0. CONFIGURACIÓN E INICIALIZACIÓN
# ==============================================================================

# '../models/...' usa los dos puntos ('../') para subir un nivel de carpetas (salir de 'homework2/') y hallar el modelo descargado.
# 'embedder' es la variable que almacena el objeto de nuestro modelo neuronal listo para codificar textos.
embedder = Embedder("../models/Xenova/all-MiniLM-L6-v2")

print("Cargando documentos desde GitHub...")

# 'GithubRepositoryDataReader' se instancia configurando las reglas para traer las lecciones.
# 'repo_owner' / 'repo_name': Indican el usuario de GitHub y el nombre del proyecto específico.
# 'commit_id': Clava una versión exacta en el tiempo de los archivos para que el dataset sea idéntico para todos.
# 'allowed_extensions': Restringe la descarga únicamente a archivos de texto con formato Markdown ('.md').
# 'filename_filter': Una función lambda (anónima) que filtra y deja pasar solo los archivos dentro de la carpeta '/lessons/'.
# 'reader' es la variable que contiene nuestro objeto extractor configurado.
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

# 'reader.read()' realiza la conexión e itera sobre los archivos remotos bajándolos a memoria.
# 'file.parse()' procesa cada archivo crudo convirtiéndolo en un diccionario limpio de propiedades (ej: texto, ruta).
# 'documents' es una lista de Python que guarda los 72 archivos Markdown ya estructurados como diccionarios.
documents = [file.parse() for file in reader.read()]
print(f"Total de páginas cargadas: {len(documents)}\n")


# ==============================================================================
# Q1. EMBEDDING DE LA CONSULTA
# ==============================================================================

# 'query_q1' es una variable de texto plano (String) que contiene la pregunta requerida por la primera consigna.
query_q1 = "How does approximate nearest neighbor search work?"

# '.encode()' agarra el string y calcula su representación numérica basándose en su significado semántico.
# 'v_query_q1' es un vector de NumPy que guarda exactamente una lista de 384 números de punto flotante.
v_query_q1 = embedder.encode(query_q1)

print(f"--- RESPUESTA Q1 ---")
# 'v_query_q1[0]' usa corchetes para extraer el número ubicado en la primera posición (índice 0) de ese vector.
# ':.2f' es un formateador de strings que reduce la salida en la consola a únicamente dos números decimales.
print(f"El primer valor (v[0]) es: {v_query_q1[0]:.2f}\n")


# ==============================================================================
# Q2. SIMILITUD DE COSENO
# ==============================================================================

# 'target_filename' es una variable de texto que define la ruta exacta de la lección solicitada por la pregunta 2.
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"

# 'next()' recorre la lista 'documents' y frena en seco en el primer elemento que cumpla la condición.
# '(doc for doc in documents if doc["filename"] == target_filename)': Busca un documento con la ruta idéntica.
# 'None': Es el valor que devolverá la función si recorre toda la lista y no encuentra ninguna coincidencia.
# 'target_doc' es la variable que almacena el diccionario con los textos de esa lección específica de SQLite.
target_doc = next((doc for doc in documents if doc["filename"] == target_filename), None)

# 'if target_doc:' verifica que la variable contenga datos válidos y no sea un elemento vacío (None).
if target_doc:
    # 'target_doc["content"]' extrae todo el cuerpo del texto escrito dentro de esa página Markdown.
    # 'v_doc' es el vector de 384 dimensiones que representa el significado conceptual de esa lección de SQLite.
    v_doc = embedder.encode(target_doc["content"])
    
    # 'np.dot()' multiplica matemáticamente dos vectores elemento por elemento y suma sus productos (producto punto).
    # Como los vectores de este modelo vienen normalizados (longitud = 1), el producto punto da exactamente la similitud de coseno.
    # 'similarity_q2' es la variable flotante que guarda el nivel de similitud conceptual entre la pregunta y la lección.
    similarity_q2 = np.dot(v_query_q1, v_doc)
    print(f"--- RESPUESTA Q2 ---")
    print(f"Similitud de coseno: {similarity_q2:.2f}\n")
else:
    print("Error: No se encontró el archivo para Q2.\n")


# ==============================================================================
# Q3. CHUNKING Y BÚSQUEDA MANUAL
# ==============================================================================
print("Aplicando chunking a los documentos...")

# 'chunk_documents()' toma tus 72 páginas largas de lecciones y las corta en bloques pequeños de caracteres.
# 'size=2000': Establece que cada fragmento tenga un tamaño máximo de 2000 caracteres.
# 'step=1000': Configura un desplazamiento de 1000 caracteres, creando un solape para evitar romper oraciones a la mitad.
# 'chunks' es una lista de Python que guarda cientos de diccionarios con fragmentos pequeños de texto.
chunks = chunk_documents(documents, size=2000, step=1000)

print("Generando embeddings para los chunks (esto puede tardar unos segundos)...")

# 'chunk_contents' es una lista de strings que extrae de forma masiva solo el campo 'content' de cada chunk, ignorando metadatos.
chunk_contents = [chunk["content"] for chunk in chunks]

# '.encode_batch()' procesa toda la lista de fragmentos en paralelo de forma eficiente usando el modelo ONNX.
# 'X' es una matriz bidimensional de NumPy (renglones y columnas). Cada fila es un chunk y sus 384 columnas son su vector.
X = embedder.encode_batch(chunk_contents)

# '.dot()' multiplica la matriz gigante 'X' por el vector unitario de tu pregunta Q1 de un solo tirón.
# Calcula de manera masiva la similitud de coseno de la pregunta contra cada uno de los fragmentos del dataset.
# 'scores' es un arreglo unidimensional de NumPy que contiene todas las puntuaciones matemáticas ordenadas por chunk.
scores = X.dot(v_query_q1)

# 'np.argmax()' inspecciona la lista de puntajes y localiza cuál posición (índice numérico) contiene el valor más alto.
# 'highest_score_idx' es una variable entera que guarda la coordenada del fragmento ganador.
highest_score_idx = np.argmax(scores)

# Usa el índice ganador para extraer el objeto original de la lista indexada de fragmentos.
# 'best_chunk' es la variable que almacena el fragmento de texto más relevante, la cual contiene la ruta buscada para Q3.
best_chunk = chunks[highest_score_idx]

print(f"--- RESPUESTA Q3 ---")
print(f"El archivo con el chunk de mayor puntuación es: {best_chunk['filename']}\n")



Cargando documentos desde GitHub...
Total de páginas cargadas: 72

--- RESPUESTA Q1 ---
El primer valor (v[0]) es: -0.02

--- RESPUESTA Q2 ---
Similitud de coseno: 0.36

Aplicando chunking a los documentos...
Generando embeddings para los chunks (esto puede tardar unos segundos)...
--- RESPUESTA Q3 ---
El archivo con el chunk de mayor puntuación es: 02-vector-search/lessons/07-sqlitesearch-vector.md



In [9]:
# ==============================================================================
# Q4. BÚSQUEDA VECTORIAL CON MINSEARCH
# ==============================================================================
print("Configurando VectorSearch...")

# 'VectorSearch' es la clase de la librería minsearch encargada de la búsqueda semántica.
# 'v_search' es la variable que almacena nuestro motor de búsqueda por vectores vacío.
v_search = VectorSearch()

# '.fit()' carga los datos dentro del motor de búsqueda vectorial.
# 'X' es la matriz de NumPy que creamos antes con los vectores de 384 números de cada chunk.
# 'chunks' es la lista con los fragmentos de texto originales de las lecciones.
# Al vincularlos, el motor asocia cada vector matemático con su texto correspondiente.
v_search.fit(X, chunks)

# 'query_q4' es una variable de texto plano que guarda la pregunta exacta exigida por el enunciado en inglés.
query_q4 = "What metric do we use to evaluate a search engine?"

# '.encode()' procesa la pregunta y la convierte en un vector numérico (lista de 384 flotantes).
# 'v_query_q4' es la variable que guarda este vector que representa matemáticamente el significado de la consulta.
v_query_q4 = embedder.encode(query_q4)

# '.search()' compara el vector de la pregunta contra la matriz de vectores 'X' mediante producto punto.
# 'num_results=1' le ordena al buscador que nos traiga únicamente el resultado más parecido de todos.
# 'results_q4' es una lista que almacena el fragmento de texto ganador encontrado por significado.
results_q4 = v_search.search(v_query_q4, num_results=1)

print(f"--- RESPUESTA Q4 ---")
# 'if results_q4:' es una estructura de control que verifica si el buscador devolvió algún resultado válido.
if results_q4:
    # 'results_q4[0]' accede al primer (y único) diccionario de resultado dentro de la lista.
    # '["filename"]' extrae la ruta del archivo de GitHub al cual pertenece ese fragmento específico.
    print(f"El archivo del primer resultado es: {results_q4[0]['filename']}\n")
else:
    print("No se encontraron resultados para Q4.\n")


# ==============================================================================
# Q5. BÚSQUEDA DE TEXTO VS BÚSQUEDA VECTORIAL
# ==============================================================================
print("Configurando Index para búsqueda de texto...")

# 'Index' es la clase de minsearch para búsquedas tradicionales por texto (coincidencia de palabras clave).
# 'text_fields=["content"]' le indica al índice que busque las palabras adentro del texto de los fragmentos.
# 'keyword_fields=[]' se deja vacío porque no estamos filtrando por categorías exactas o metadatos rígidos.
# 'text_index' es la variable que almacena este motor de búsqueda clásico por palabras clave.
text_index = Index(text_fields=["content"], keyword_fields=[])

# '.fit()' escanea todos nuestros fragmentos de texto y construye un índice invertido interno.
# 'chunks' es la lista de fragmentos que ingresamos para mapear qué palabra aparece en qué lugar.
text_index.fit(chunks)

# 'query_q5' es una variable de texto que guarda la consulta exacta de la pregunta 5.
query_q5 = "How do I store vectors in PostgreSQL?"

print("Ejecutando búsqueda por texto...")
# '.search()' busca en el índice invertido los fragmentos que contengan las palabras exactas de la consulta.
# 'num_results=5' le pide al buscador tradicional que nos devuelva el top 5 de coincidencias.
# 'text_results' es una lista que almacena los 5 fragmentos encontrados mediante texto plano.
text_results = text_index.search(query_q5, num_results=5)

# Esta línea usa una "comprensión de conjuntos" (set comprehension). 
# Recorre cada documento 'doc' en 'text_results' y extrae el valor de la clave 'filename'.
# 'text_filenames' es un Conjunto (Set) de Python que guarda los nombres únicos de archivos hallados por texto.
text_filenames = {doc["filename"] for doc in text_results}

print("Ejecutando búsqueda vectorial...")
# '.encode()' transforma la consulta de texto de Q5 en su respectivo vector matemático de 384 dimensiones.
# 'v_query_q5' es la variable que almacena ese vector de significado de la pregunta Q5.
v_query_q5 = embedder.encode(query_q5)

# Usamos nuestro motor 'v_search' (ya configurado) para buscar fragmentos con significado similar.
# 'num_results=5' le pide al buscador vectorial que extraiga los 5 mejores resultados conceptuales.
# 'vector_results' es una lista que almacena los 5 mejores fragmentos encontrados por embeddings.
vector_results = v_search.search(v_query_q5, num_results=5)

# Volvemos a usar una comprensión de conjuntos para extraer las rutas de los archivos vectoriales.
# 'vector_filenames' es un Conjunto (Set) de Python que almacena los nombres de archivos hallados por vectores.
vector_filenames = {doc["filename"] for doc in vector_results}

# El operador '-' ejecuta una resta matemática entre conjuntos lógicos.
# Toma el conjunto 'vector_filenames' y elimina cualquier nombre de archivo que también esté en 'text_filenames'.
# 'diff' es la variable que guarda los archivos que la búsqueda vectorial descubrió por concepto pero que la búsqueda de texto no vio.
diff = vector_filenames - text_filenames

print(f"--- RESPUESTA Q5 ---")
# Imprime el contenido de 'diff'. Verás el nombre del archivo exacto para marcar en tu examen.
print(f"Archivos en vector pero NO en texto: {diff}\n")


Configurando VectorSearch...
--- RESPUESTA Q4 ---
El archivo del primer resultado es: 04-evaluation/lessons/05-search-metrics.md

Configurando Index para búsqueda de texto...
Ejecutando búsqueda por texto...
Ejecutando búsqueda vectorial...
--- RESPUESTA Q5 ---
Archivos en vector pero NO en texto: {'02-vector-search/lessons/08-pgvector.md'}



In [11]:
# ==============================================================================
# P6. BÚSQUEDA HÍBRIDA CON RECIPROCAL RANK FUSION (RRF)
# ==============================================================================

# 1. Definición de la función matemática RRF suministrada por la cátedra
# 'result_lists' es una lista que contiene sublistas de resultados (ej: [lista_vectorial, lista_texto]).
# 'k=60' es la constante de suavizado estándar del algoritmo para que los primeros puestos no aplasten por completo a los demás.
# 'num_results=5' es la cantidad máxima de documentos fusionados que queremos que nos devuelva la función.
def rrf(result_lists, k=60, num_results=5):
    
    # 'scores' es un diccionario vacío que guardará pares de {clave_única: puntaje_numérico_acumulado}.
    scores = {}
    
    # 'docs' es un diccionario vacío que guardará pares de {clave_única: diccionario_completo_del_chunk}.
    # Lo usamos para poder recuperar el texto y metadatos originales al final del proceso.
    docs = {}

    # 'for results in result_lists:' es un bucle que recorre las listas de búsqueda una por una.
    # Primero procesará toda la lista de resultados vectoriales y, en la siguiente vuelta, toda la lista de texto.
    for results in result_lists:
        
        # 'enumerate(results)' nos permite recorrer la lista obteniendo dos datos a la vez:
        # 'rank': La posición en el ranking (0 para el primero, 1 para el segundo, etc.).
        # 'doc': El fragmento de documento (diccionario) que está en esa posición.
        for rank, doc in enumerate(results):
            
            # 'key' es una tupla que combina el nombre del archivo y su carácter de inicio en el texto.
            # Al combinar ambos datos aseguramos un identificador único para cada fragmento, evitando confusiones si un archivo tiene varios chunks.
            key = (doc["filename"], doc["start"])
            
            # 'scores.get(key, 0)' busca si este fragmento ya tiene un puntaje previo. Si no existe, arranca en 0.
            # '1 / (k + rank)' aplica la fórmula matemática RRF: divide 1 por (60 + la posición en el ranking).
            # El resultado se suma al puntaje anterior. Si un documento aparece en ambas listas, su puntaje final será más alto.
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            
            # Guardamos el objeto completo 'doc' en nuestro diccionario usando la tupla 'key' como casillero.
            docs[key] = doc

    # 'sorted()' ordena las claves únicas del diccionario 'scores'.
    # 'key=scores.get' le indica que ordene guiándose por los puntajes numéricos acumulados en cada clave.
    # 'reverse=True' asegura que el ordenamiento sea descendente (el puntaje más alto primero).
    # 'ranked' es una lista que contiene las tuplas de identificación ordenadas de mejor a peor.
    ranked = sorted(scores, key=scores.get, reverse=True)
    
    # 'ranked[:num_results]' recorta la lista ordenada para quedarnos solo con las primeras 5 tuplas de identificación.
    # '[docs[key] for key in ...]' es una comprensión de listas que cambia las tuplas por sus diccionarios de documentos originales.
    return [docs[key] for key in ranked[:num_results]]


print("Ejecutando búsqueda híbrida...")

# 'query_p6' almacena la consulta textual en inglés que usaremos para poner a prueba los motores.
query_p6 = "How do I give the model access to tools?"

# --- COMPONENTE 1: BÚSQUEDA TRADICIONAL POR TEXTO (KEYWORD) ---
# '.search()' ejecuta la búsqueda exacta por palabras clave en nuestro índice de texto clásico 'text_index'.
# 'num_results=5' extrae las 5 mejores coincidencias literales de palabras.
# 'text_results_p6' es una lista que almacena esos 5 fragmentos hallados por texto.
text_results_p6 = text_index.search(query_p6, num_results=5)

# --- COMPONENTE 2: BÚSQUEDA CON VECTORES (SEMÁNTICA) ---
# '.encode()' traduce nuestra consulta de la pregunta 6 en un vector matemático de 384 dimensiones.
# 'v_query_p6' almacena este embedding semántico de la consulta.
v_query_p6 = embedder.encode(query_p6)

# '.search()' analiza conceptualmente la matriz usando nuestro motor vectorial 'v_search'.
# 'num_results=5' extrae los 5 fragmentos que tienen el significado más cercano a la pregunta.
# 'vector_results_p6' es una lista que almacena esos 5 fragmentos hallados por vectores.
vector_results_p6 = v_search.search(v_query_p6, num_results=5)


# --- FUSIÓN HÍBRIDA (RRF) ---
# Invocamos nuestra función 'rrf' pasándole una lista con los dos bloques de 5 resultados obtenidos.
# 'hybrid_results' guarda la lista final unificada de 5 fragmentos, reordenados según la fuerza de sus posiciones combinadas.
hybrid_results = rrf([vector_results_p6, text_results_p6])

print(f"--- RESPUESTA P6 ---")
# 'if hybrid_results:' verifica que la lista combinada final contenga elementos y no esté vacía.
if hybrid_results:
    # 'hybrid_results[0]' accede al primer elemento de la lista fusionada (el campeón absoluto de la búsqueda híbrida).
    # '["filename"]' extrae la propiedad que indica a qué lección pertenece ese fragmento.
    print(f"El archivo que clasifica primero después de RRF es: {hybrid_results[0]['filename']}\n")
else:
    print("No se encontraron resultados híbridos.\n")


Ejecutando búsqueda híbrida...
--- RESPUESTA P6 ---
El archivo que clasifica primero después de RRF es: 01-agentic-rag/lessons/13-function-calling.md

